# Project 6 — Marketing Attribution & Campaign Performance

**Student:** Sachchidanand  |  **Section:** AI ML-5  |  **Admission Number:** 2026B15310170

**Objective:** compare campaign and channel efficiency, explain attribution assumptions, and model a constrained budget scenario.

> Dataset is synthetic. The source contains last-touch campaign assignment only. First-touch and linear journeys below are simulated solely to demonstrate attribution sensitivity, not observed customer behavior.

Keep the four source CSVs in `data/` beside this notebook. In Colab, upload the package ZIP, extract it, then set `DATA_DIR` to the extracted `data` folder.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
DATA_DIR=Path('data')
files={n:pd.read_csv(DATA_DIR/f'{n}.csv') for n in ['campaigns','daily_spend','leads','conversions']}
c,d,l,v=(files[n] for n in ['campaigns','daily_spend','leads','conversions'])
print({k:x.shape for k,x in files.items()})

## 1. Inspect and validate the source tables
The tables have different grains. Review types, date coverage, missingness, duplicate keys, and relationships before joining.

In [ ]:
for name,df in files.items():
 print('\n',name,df.shape); display(df.head()); display(df.info()); display(df.describe(include='all').T); print('missing',df.isna().sum().sum(),'duplicate rows',df.duplicated().sum())
print('Duplicate IDs:',c.campaign_id.duplicated().sum(),d.spend_id.duplicated().sum(),l.lead_id.duplicated().sum(),v.conversion_id.duplicated().sum())
print('Unknown campaign refs:',(~d.campaign_id.isin(c.campaign_id)).sum(),(~l.campaign_id.isin(c.campaign_id)).sum())
print('Unknown conversion lead refs:',(~v.lead_id.isin(l.lead_id)).sum())

## 2. Clean and join without multiplying spend

In [ ]:
for df,cols in [(c,['start_date','end_date']),(d,['spend_date']),(l,['created_at']),(v,['converted_at'])]:
 for col in cols: df[col]=pd.to_datetime(df[col],errors='coerce')
for df,col in [(c,'channel'),(d,'campaign_id'),(l,'campaign_id'),(l,'lead_source'),(v,'lead_id')]: df[col]=df[col].astype('string').str.strip()
sp=d.groupby('campaign_id',as_index=False).agg(impressions=('impressions','sum'),clicks=('clicks','sum'),spend=('spend','sum'),active_days=('spend_date','nunique'))
la=l.groupby('campaign_id',as_index=False).agg(leads=('lead_id','nunique'),mean_lead_score=('lead_score','mean'))
vl=v.merge(l[['lead_id','campaign_id']],on='lead_id',how='left',validate='one_to_one',indicator=True)
va=vl.groupby('campaign_id',as_index=False).agg(conversions=('conversion_id','nunique'),revenue=('revenue','sum'),purchases=('conversion_type',lambda s:(s=='purchase').sum()),trials=('conversion_type',lambda s:(s=='trial').sum()))
p=c.merge(sp,on='campaign_id',how='left',validate='one_to_one').merge(la,on='campaign_id',how='left',validate='one_to_one').merge(va,on='campaign_id',how='left',validate='one_to_one').fillna(0)
print('Spend check:',d.spend.sum(),p.spend.sum(),'Revenue check:',v.revenue.sum(),p.revenue.sum())

## 3. Calculate KPIs by campaign and channel
CTR = clicks/impressions; CPC = spend/clicks; CVR = conversions/clicks; CPA = spend/conversions; ROAS = attributed revenue/spend. Rates use totals, not average row-level rates.

In [ ]:
def safe_divide(a,b):
 a=np.asarray(a,dtype=float); b=np.asarray(b,dtype=float); return np.divide(a,b,out=np.full(np.broadcast(a,b).shape,np.nan),where=b!=0)
p['ctr']=safe_divide(p.clicks,p.impressions); p['cpc']=safe_divide(p.spend,p.clicks); p['cvr']=safe_divide(p.conversions,p.clicks); p['cpa']=safe_divide(p.spend,p.conversions); p['roas']=safe_divide(p.revenue,p.spend)
ch=p.groupby('channel',as_index=False).agg(spend=('spend','sum'),budget=('budget','sum'),impressions=('impressions','sum'),clicks=('clicks','sum'),leads=('leads','sum'),conversions=('conversions','sum'),revenue=('revenue','sum'),campaigns=('campaign_id','nunique'))
for m,a,b in [('ctr','clicks','impressions'),('cpc','spend','clicks'),('cvr','conversions','clicks'),('cpa','spend','conversions'),('roas','revenue','spend')]: ch[m]=ch[a]/ch[b]
display(ch.sort_values('roas',ascending=False).round(3))

## 4. Bootstrap uncertainty and attribution sensitivity
Bootstrap resamples campaigns within channel. Since multi-touch journeys are absent, simulate 1–3 prior channel touches with a fixed seed; only the campaign assigned in the source remains the final touch. These alternative credit models are illustrative.

In [ ]:
rng=np.random.default_rng(42); boots=[]
for name,g in p.groupby('channel'):
 x=g.roas.dropna().to_numpy(); z=rng.choice(x,size=(3000,len(x)),replace=True).mean(axis=1); boots.append({'channel':name,'mean':x.mean(),'median':np.median(x),'lo':np.quantile(z,.025),'hi':np.quantile(z,.975)})
bootstrap=pd.DataFrame(boots); display(bootstrap.round(3))
journeys=vl.merge(c[['campaign_id','channel']],on='campaign_id',validate='many_to_one'); channels=c.channel.drop_duplicates().tolist()
prior_n=rng.choice([1,2,3],size=len(journeys),p=[.5,.35,.15]); first=[]; credits=[]
for i,row in enumerate(journeys.itertuples(index=False)):
 path=list(rng.choice([x for x in channels if x!=row.channel],size=int(prior_n[i]),replace=True))+[row.channel]; first.append(path[0]); credits.extend((i,chx,row.revenue/len(path)) for chx in path)
journeys['sim_first']=first
last=journeys.groupby('channel').revenue.sum().rename('credit').reset_index(); last['model']='Observed last-touch'
first_df=journeys.groupby('sim_first').revenue.sum().rename('credit').reset_index().rename(columns={'sim_first':'channel'}); first_df['model']='Simulated first-touch'
linear=pd.DataFrame(credits,columns=['conversion','channel','credit']).groupby('channel',as_index=False).credit.sum(); linear['model']='Simulated linear'
attr=pd.concat([last,first_df,linear]).merge(ch[['channel','spend']],on='channel',validate='many_to_one'); attr['roas']=attr.credit/attr.spend
print('First-touch/linear paths are generated, not observed.')

## 5. Time analysis and budget scenario

In [ ]:
monthly=d.set_index('spend_date').groupby('campaign_id').resample('MS')[['spend','clicks','impressions']].sum().reset_index().groupby('spend_date',as_index=False).agg(spend=('spend','sum'),clicks=('clicks','sum'),impressions=('impressions','sum'))
monthly['ctr']=monthly.clicks/monthly.impressions; monthly['spend_3m']=monthly.spend.rolling(3,min_periods=1).mean()
video=ch.loc[ch.channel.eq('video'),'spend'].iloc[0]; moved=.10*video; scenario=ch[['channel','spend','revenue','roas']].copy(); scenario['scenario_spend']=scenario.spend; scenario.loc[scenario.channel.eq('video'),'scenario_spend']-=moved
targets=ch[ch.channel.isin(['affiliate','paid_search'])]; weights=targets.spend/targets.spend.sum()
for row,w in zip(targets.itertuples(index=False),weights): scenario.loc[scenario.channel.eq(row.channel),'scenario_spend']+=moved*w
scenario['scenario_revenue']=scenario.scenario_spend*scenario.roas; scenario['delta']=scenario.scenario_revenue-scenario.revenue; display(scenario.round(2))
print('Estimated delta under unchanged historical ROAS:',scenario.delta.sum())

## 6. Five visualizations

In [ ]:
fig,ax=plt.subplots(2,3,figsize=(16,9)); ax=ax.ravel(); q=ch.sort_values('roas',ascending=False)
ax[0].bar(q.channel,q.roas,color='#2463A6'); ax[0].set_title('Attributed ROAS by channel')
q=ch.sort_values('cpa'); ax[1].bar(q.channel,q.cpa,color='#C66A3D'); ax[1].set_title('Cost per conversion')
q=ch.sort_values('ctr'); ax[2].bar(q.channel,q.ctr*100,color='#3F8F75'); ax[2].set_title('CTR (%)')
ax[3].plot(monthly.spend_date,monthly.spend,label='Spend'); ax[3].plot(monthly.spend_date,monthly.spend_3m,label='3-month average'); ax[3].set_title('Monthly spend'); ax[3].legend()
ax[4].boxplot([p.loc[p.channel.eq(x),'roas'].dropna() for x in ch.channel],tick_labels=ch.channel); ax[4].set_title('Campaign ROAS distribution')
attr.pivot(index='channel',columns='model',values='credit').plot(kind='bar',ax=ax[5]); ax[5].set_title('Attribution credit sensitivity'); plt.tight_layout(); plt.show()

## 7. Findings and limitations

In [ ]:
print('\n'.join(f'{i}. {x}' for i,x in enumerate(['Total media spend was $4,523,050 and last-touch attributed revenue was $21,561,839, for aggregate ROAS of 4.77x.', 'affiliate had the highest channel ROAS at 6.54x; video was lowest at 2.52x.', 'paid_search had the highest CTR (3.49%); click response and revenue efficiency rank channels differently.', 'affiliate had the lowest CPA ($82.87); video had the highest ($210.55).', 'paid_social received the most spend ($984,732) but did not lead on ROAS.', '39,696 conversions came from 261,684 leads; overall lead-to-conversion rate was 15.17%.', 'Mean campaign ROAS bootstrap 95% interval for affiliate was 6.52x–9.28x; it describes synthetic campaign variation, not causal uncertainty.', 'The source has no observed multi-touch journeys; first-touch/linear attribution is simulated only.', 'A scenario shifting 10% of video spend ($82,809) estimates $312,661 additional attributed revenue only if historical ROAS holds constant.'],1)))

All records are synthetic educational data. Last-touch assignment is not causal lift. Simulated journeys only demonstrate rule sensitivity. Revenue is not profit; margin, returns, lifetime value, overlap, and saturation are not supplied. The budget scenario holds historical ROAS constant. A real recommendation requires actual timestamped touches and randomized holdout or geo testing.